<a href="https://colab.research.google.com/github/am-3/IB9AU-2026/blob/main/Task3_German_Credit_Risk_Prediction_with_PyTorchMLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Task 3: German Credit Risk Prediction with PyTorch MLP
___
### Objective
This notebook addresses the critical FinTech problem of **credit risk prediction**. The goal is to build, train, and evaluate a Multi-Layer Perceptron (MLP) model using PyTorch to predict whether a bank customer is likely to default on a loan, based on the "German Credit Data" dataset.
___
### Tech Stack
*   **Data Manipulation & Preprocessing**: `pandas`, `numpy`, `scikit-learn`
*   **Machine Learning Framework**: `PyTorch` (`torch.nn`, `torch.optim`, `torch.utils.data`)
___
### Methodology
The end-to-end workflow involves several key stages:
1.  **Data Loading & Preprocessing**: The German Credit Data was loaded, column names were assigned, and the target variable (`Creditability`) was remapped for binary classification. Categorical features were transformed using one-hot encoding, and numerical features were standardized using `StandardScaler` (fitted only on training data to prevent data leakage). The dataset was split into 80% training and 20% testing sets.
2.  **PyTorch Data Setup**: Processed data was converted into PyTorch `FloatTensors`, and `TensorDataset` and `DataLoader` instances were created for efficient batching during training and evaluation.
3.  **Model Architecture**: A custom MLP was defined using `torch.nn.Module`, featuring two hidden layers with ReLU activation functions and a final output layer with a Sigmoid activation for binary classification probabilities.
4.  **Model Training**: The model was trained for 50 epochs using `nn.BCELoss` as the criterion and `optim.Adam` as the optimizer, iteratively minimizing the loss over batches of training data.
5.  **Model Evaluation**: The trained model's performance was assessed on the unseen test set, reporting overall accuracy, a detailed classification report (precision, recall, f1-score), and a confusion matrix to understand prediction strengths and weaknesses.
___

## Step 1: Data Loading & Preprocessing (Pandas & Scikit-Learn)

This crucial initial step involves preparing our raw data for machine learning. We'll use `pandas` for data manipulation and `scikit-learn` for splitting and scaling.

### 1.1 Load Data
We'll fetch the dataset directly from the UCI ML Repository. Since it's space-separated and lacks a header, we specify `sep='\s+'` and `header=None`.

### 1.2 Assign Columns
To make the data understandable, we'll assign meaningful column names as provided.

### 1.3 Target Mapping
The original 'Creditability' column has values 1 (Good) and 2 (Bad). For PyTorch's `BCELoss`, we need binary targets 0 and 1. We'll map 1 to 1 (Good) and 2 to 0 (Bad).

### 1.4 Categorical Encoding
Machine learning models, especially neural networks, work best with numerical input. We'll identify all categorical features and convert them into numerical representations using one-hot encoding (`pd.get_dummies`). This creates new binary columns for each category level.

### 1.5 Train/Test Split
To evaluate our model's generalization ability, we split the dataset into an 80% training set and a 20% testing set. It's vital to `stratify` the split by the target variable (`y`) to ensure that both training and testing sets have a similar proportion of good and bad credit instances, preventing biased evaluation.

### 1.6 Scaling (Crucial for Neural Networks)
Neural networks are sensitive to the scale of input features. We use `StandardScaler` to transform numerical features so they have zero mean and unit variance. The formula for standardization is given by $z = \frac{x - \mu}{\sigma}$, where $x$ is the original feature value, $\mu$ is the mean, and $\sigma$ is the standard deviation. **Preventing Data Leakage**: It's paramount that the `StandardScaler` is `fit` *only* on the training data. This means the mean and standard deviation used for scaling are derived solely from the training set. We then use this *fitted* scaler to `transform` both the training and testing sets. If we fit the scaler on the entire dataset before splitting, information about the test set's distribution would leak into the training process, leading to an overly optimistic performance estimate.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np

# --- Step 1: Data Loading & Preprocessing ---

# 1.1 Load Data
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data"
df = pd.read_csv(url, sep=r'\s+', header=None)

# 1.2 Assign Columns
column_names = [
    "Status of existing checking account", "Duration in month", "Credit history",
    "Purpose", "Credit amount", "Savings account/bonds", "Present employment since",
    "Installment rate in percentage of disposable income", "Personal status and sex",
    "Other debtors / guarantors", "Present residence since", "Property",
    "Age in years", "Other installment plans", "Housing",
    "Number of existing credits at this bank", "Job",
    "Number of people being liable to provide maintenance for", "Telephone",
    "foreign worker", "Creditability"
]
df.columns = column_names

print("Original DataFrame head:\n", df.head())
print("\nDataFrame info:\n")
df.info()

Original DataFrame head:
   Status of existing checking account  Duration in month Credit history  \
0                                 A11                  6            A34   
1                                 A12                 48            A32   
2                                 A14                 12            A34   
3                                 A11                 42            A32   
4                                 A11                 24            A33   

  Purpose  Credit amount Savings account/bonds Present employment since  \
0     A43           1169                   A65                      A75   
1     A43           5951                   A61                      A73   
2     A46           2096                   A61                      A74   
3     A42           7882                   A61                      A74   
4     A40           4870                   A61                      A73   

   Installment rate in percentage of disposable income  \
0             

In [ ]:
# 1.3 Target Mapping
# Original: 1 (Good), 2 (Bad). Map to: 1 (Good), 0 (Bad).
df['Creditability'] = df['Creditability'].map({1: 1, 2: 0})
print("\nTarget distribution after mapping:\n", df['Creditability'].value_counts())

# Separate features (X) and target (y)
X = df.drop('Creditability', axis=1)
y = df['Creditability']

# Identify categorical and numerical columns
# Based on UCI dataset description and common sense for German Credit Data
categorical_cols = [
    "Status of existing checking account", "Credit history", "Purpose",
    "Savings account/bonds", "Present employment since", "Personal status and sex",
    "Other debtors / guarantors", "Property", "Other installment plans", "Housing",
    "Job", "Telephone", "foreign worker"
]

numerical_cols = [
    "Duration in month", "Credit amount",
    "Installment rate in percentage of disposable income",
    "Present residence since", "Age in years",
    "Number of existing credits at this bank",
    "Number of people being liable to provide maintenance for"
]

# Ensure all identified columns are actually in the DataFrame
categorical_cols = [col for col in categorical_cols if col in X.columns]
numerical_cols = [col for col in numerical_cols if col in X.columns]

# 1.4 Categorical Encoding using pd.get_dummies
X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True) # drop_first to avoid multicollinearity

# Ensure all numerical columns are included in X_encoded even if they weren't dummified
X_processed = pd.concat([X_encoded[numerical_cols], X_encoded.drop(columns=numerical_cols)], axis=1)

print("\nFeatures after one-hot encoding (first 5 rows):\n", X_processed.head())
print("Shape of features after encoding:", X_processed.shape)



Target distribution after mapping:
 Creditability
1    700
0    300
Name: count, dtype: int64

Features after one-hot encoding (first 5 rows):
    Duration in month  Credit amount  \
0                  6           1169   
1                 48           5951   
2                 12           2096   
3                 42           7882   
4                 24           4870   

   Installment rate in percentage of disposable income  \
0                                                  4     
1                                                  2     
2                                                  2     
3                                                  2     
4                                                  3     

   Present residence since  Age in years  \
0                        4            67   
1                        2            22   
2                        3            49   
3                        4            45   
4                        4            53   

   Num

In [ ]:
# 1.5 Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTraining features shape: {X_train.shape}")
print(f"Testing features shape: {X_test.shape}")
print(f"Training target shape: {y_train.shape}")
print(f"Testing target shape: {y_test.shape}")

# 1.6 Scaling Numerical Features
scaler = StandardScaler()

# Crucial: Fit scaler ONLY on training data to prevent data leakage
X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])

# Transform both training and testing data using the fitted scaler
X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])

print("\nTraining features after scaling (first 5 rows):\n", X_train.head())
print("Testing features after scaling (first 5 rows):\n", X_test.head())



Training features shape: (800, 48)
Testing features shape: (200, 48)
Training target shape: (800,)
Testing target shape: (200,)

Training features after scaling (first 5 rows):
      Duration in month  Credit amount  \
675           0.755149       0.485384   
703           0.755149      -0.246578   
12           -0.726746      -0.584573   
845           0.014201       0.285331   
795          -0.973728      -0.319522   

     Installment rate in percentage of disposable income  \
675                                           0.905268     
703                                           0.905268     
12                                           -1.797024     
845                                          -0.896260     
795                                          -0.896260     

     Present residence since  Age in years  \
675                 1.044365     -0.825479   
703                -0.758207      0.493705   
12                 -1.659492     -1.177262   
845                 0.143079 

___
## Step 2: PyTorch Data Setup


PyTorch models expect data in specific tensor formats. This step involves converting our preprocessed pandas DataFrames/NumPy arrays into PyTorch `FloatTensors`. We then wrap these tensors into `TensorDataset` objects, which pair features with their corresponding labels. Finally, `DataLoader`s are created to handle batching, shuffling (for training), and loading data efficiently into the model during training and evaluation. A batch size of 64 is chosen to balance computational efficiency and convergence stability.

In [ ]:
# --- Step 2: PyTorch Data Setup ---

# Convert to PyTorch FloatTensors (explicitly cast NumPy array to float32)
X_train_tensor = torch.tensor(X_train.values.astype(np.float32), dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1) # unsqueeze for BCELoss
X_test_tensor = torch.tensor(X_test.values.astype(np.float32), dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).unsqueeze(1)

# Create TensorDatasets
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

# Create DataLoaders
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"\nTrain TensorDataset size: {len(train_dataset)}")
print(f"Test TensorDataset size: {len(test_dataset)}")
print(f"Batch size: {batch_size}")



Train TensorDataset size: 800
Test TensorDataset size: 200
Batch size: 64


___
## Step 3: Model Architecture

Here, we define our Multi-Layer Perceptron (MLP) using PyTorch's `nn.Module`. The architecture consists of multiple fully connected (linear) layers and activation functions.

*   **Input Dimension**: This is determined by the number of features after one-hot encoding and scaling (`X_train.shape[1]`).
*   **Hidden Layers**: We use two hidden layers with `nn.Linear` transformations. `ReLU` (`nn.ReLU`) is chosen as the activation function for hidden layers because it introduces non-linearity, allowing the model to learn complex patterns, and helps mitigate the vanishing gradient problem. A typical neuron's activation can be represented as $a = f(W \cdot x + b)$, where $W$ is the weight matrix, $x$ is the input, $b$ is the bias, and $f$ is the activation function (e.g., ReLU).
*   **Output Layer**: For binary classification, the output layer has a single neuron. The `Sigmoid` function (`nn.Sigmoid`) is applied to this output. Sigmoid squashes the output into a range between 0 and 1, which can be interpreted as the probability of the positive class (in our case, good credit).

In [ ]:
# --- Step 3: Model Architecture ---

# Define the MLP model
class MLP(nn.Module):
    def __init__(self, input_dim):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(input_dim, 128)  # First hidden layer
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(128, 64)   # Second hidden layer
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(64, 1)    # Output layer
        self.sigmoid = nn.Sigmoid()    # Sigmoid activation for binary classification

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.fc3(x)
        x = self.sigmoid(x)
        return x

# Determine input dimension based on preprocessed features
input_dim = X_train.shape[1]
model = MLP(input_dim)

print("\nMLP Model Architecture:\n", model)



MLP Model Architecture:
 MLP(
  (fc1): Linear(in_features=48, out_features=128, bias=True)
  (relu1): ReLU()
  (fc2): Linear(in_features=128, out_features=64, bias=True)
  (relu2): ReLU()
  (fc3): Linear(in_features=64, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)


___
## Step 4: Training Loop

This is where the model learns from the training data. We define the loss function and the optimizer, then iterate through the data for a specified number of epochs.

*   **Loss Function**: `nn.BCELoss()` (Binary Cross-Entropy Loss) is chosen because it's standard for binary classification problems when the model's output is a probability (after a sigmoid activation).
*   **Optimizer**: `optim.Adam` is an adaptive learning rate optimization algorithm known for its efficiency and good performance in many deep learning tasks.
*   **Training Process**: For each epoch, we iterate over batches from the `train_loader`. In each iteration:
    1.  **Forward Pass**: Input data passes through the model to get predictions.
    2.  **Calculate Loss**: The predictions are compared to the true labels using the loss function.
    3.  **Zero Gradients**: Gradients from the previous iteration are cleared.
    4.  **Backward Pass**: Gradients of the loss with respect to model parameters are computed.
    5.  **Optimizer Step**: Model parameters are updated using the gradients to minimize the loss.
*   **Monitoring**: Training loss is printed periodically to observe the model's learning progress.

In [ ]:
# --- Step 4: Training Loop ---

# Initialize model, loss function, and optimizer
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 50

print("\nStarting Training...")
for epoch in range(num_epochs):
    model.train() # Set model to training mode
    running_loss = 0.0
    for inputs, labels in train_loader:
        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)

    epoch_loss = running_loss / len(train_dataset)

    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}')

print("Training Finished!")



Starting Training...
Epoch [1/50], Loss: 0.6430
Epoch [10/50], Loss: 0.4252
Epoch [20/50], Loss: 0.2737
Epoch [30/50], Loss: 0.1024
Epoch [40/50], Loss: 0.0352
Epoch [50/50], Loss: 0.0136
Training Finished!


___
## Step 5: Evaluation

After training, we evaluate the model's performance on the unseen test set. This step assesses how well the model generalizes to new data.

*   **`torch.no_grad()`**: We use `torch.no_grad()` to disable gradient calculations during evaluation. This saves memory and computation, as we only need predictions, not updates to the model parameters.
*   **Predictions**: The model is set to `eval()` mode (which disables dropout, batch normalization updates, etc., if present). We iterate through the `test_loader` to get predictions. A threshold of 0.5 is applied to the sigmoid outputs to classify instances as 0 (bad credit) or 1 (good credit).
*   **Metrics**: We calculate:
    *   **Accuracy**: The proportion of correctly classified instances.
    *   **Classification Report**: Provides precision, recall, f1-score, and support for each class.
    *   **Confusion Matrix**: Shows the counts of true positive, true negative, false positive, and false negative predictions, offering a detailed view of the model's performance across classes.

In [ ]:
# --- Step 5: Evaluation ---

model.eval() # Set model to evaluation mode

all_predictions = []
all_labels = []

with torch.no_grad(): # Disable gradient computation for evaluation
    for inputs, labels in test_loader:
        outputs = model(inputs)
        # Convert probabilities to binary predictions (0 or 1)
        predictions = (outputs >= 0.5).float()

        all_predictions.extend(predictions.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Convert lists to numpy arrays for scikit-learn metrics
all_predictions = np.array(all_predictions).flatten()
all_labels = np.array(all_labels).flatten()

# Calculate and print accuracy
accuracy = accuracy_score(all_labels, all_predictions)
print(f"\nModel Evaluation on Test Set:\n")
print(f"Accuracy: {accuracy:.4f}")

# Print classification report
print("\nClassification Report:\n")
print(classification_report(all_labels, all_predictions, target_names=['Bad Credit (0)', 'Good Credit (1)']))

# Print confusion matrix
conf_matrix = confusion_matrix(all_labels, all_predictions)
print("\nConfusion Matrix:\n")
print(conf_matrix)

# Interpretation of Confusion Matrix:
# [[TN, FP]
#  [FN, TP]]
# TN = True Negatives (Correctly predicted Bad Credit)
# FP = False Positives (Incorrectly predicted Good Credit - Type I error)
# FN = False Negatives (Incorrectly predicted Bad Credit - Type II error)
# TP = True Positives (Correctly predicted Good Credit)



Model Evaluation on Test Set:

Accuracy: 0.7100

Classification Report:

                 precision    recall  f1-score   support

 Bad Credit (0)       0.51      0.60      0.55        60
Good Credit (1)       0.82      0.76      0.79       140

       accuracy                           0.71       200
      macro avg       0.66      0.68      0.67       200
   weighted avg       0.73      0.71      0.72       200


Confusion Matrix:

[[ 36  24]
 [ 34 106]]


___
## Insights & Learnings

### Key Results
*   **Overall Accuracy**: The MLP model achieved an accuracy of **71.00%** on the unseen test set.
*   **Class-wise Performance**:
    *   **Bad Credit (0)**: The model showed a precision of 51% and recall of 60%. This indicates that when the model predicts a customer has 'Bad Credit', it's correct about half the time. It also correctly identifies 60% of actual 'Bad Credit' cases.
    *   **Good Credit (1)**: The model achieved a higher precision of 82% and recall of 76%. This suggests the model is more reliable when predicting 'Good Credit' and identifies a good proportion of actual 'Good Credit' cases.
*   **Confusion Matrix Breakdown**:
    *   **True Positives (TP)**: 106 (Correctly identified 'Good Credit')
    *   **True Negatives (TN)**: 36 (Correctly identified 'Bad Credit')
    *   **False Positives (FP)**: 24 (Incorrectly predicted 'Good Credit' when it was 'Bad Credit' - Type I error)
    *   **False Negatives (FN)**: 34 (Incorrectly predicted 'Bad Credit' when it was 'Good Credit' - Type II error)
___
### Technical Learnings
*   **Data Leakage Prevention**: This task reinforced the critical importance of fitting `StandardScaler` *only* on the training data and then transforming both training and testing sets. Failing to do so would lead to overly optimistic performance metrics due to information leakage from the test set's distribution.
*   **PyTorch for Structured Data**: While often associated with image or natural language processing, PyTorch provides a robust framework for building and training neural networks on structured tabular data. The use of `TensorDataset` and `DataLoader` significantly streamlined data management, batching, and shuffling.
*   **Binary Classification Setup**: The combination of `nn.BCELoss` for the loss function and a `nn.Sigmoid` activation in the output layer is standard and effective for binary classification tasks, allowing the model to output probabilities directly.
*   **Feature Engineering Impact**: The performance of the MLP is heavily reliant on the initial preprocessing, particularly one-hot encoding for categorical features and standardization for numerical ones. Without these steps, the model would struggle to converge or learn meaningful patterns.
___
### Practical Application
This credit risk prediction model has direct applicability in the FinTech sector. Banks and financial institutions can leverage such models to:
*   **Automate Loan Approvals**: Quickly assess the creditworthiness of applicants, speeding up decision-making processes.
*   **Reduce Financial Risk**: By accurately identifying high-risk borrowers, banks can mitigate potential loan defaults and minimize financial losses.
*   **Optimize Interest Rates**: Offer differentiated interest rates based on predicted risk, thereby attracting more reliable customers and managing risk effectively.
*   **Fraud Detection**: The underlying principles can be extended to detect fraudulent transactions by identifying unusual patterns associated with high-risk behavior.
___
The model's tendency for higher precision on 'Good Credit' predictions is beneficial for avoiding false rejections of creditworthy customers, while the recall on 'Bad Credit' is crucial for minimizing defaults. Further improvements could focus on optimizing the balance between precision and recall, potentially through weighted loss functions or adjusting the classification threshold, depending on the business's risk appetite.